<a href="https://colab.research.google.com/github/irfftny/codeinhcl/blob/master/Tarik_Subtitle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 55.7 MB/s eta 0:00:00


In [3]:
!pip install gspread oauth2client pandas

In [4]:
!pip install yt-dlp pandas openpyxl

In [5]:
!pip install yt-dlp gspread

In [7]:
import yt_dlp
import re
import os
from datetime import timedelta
from google.colab import auth
import gspread
from google.auth import default

# ================= PENGATURAN =================
link_youtube = "https://www.youtube.com/live/5iW_mfLSSW4?si=kMCcsIZI37J700tr"
nama_sheet_tujuan = "Kajian_Santai_Fiqh_Tasawuf"
nama_file_sementara = "temp_sub"
# ==============================================

def waktu_ke_detik(waktu_str):
    waktu_str = waktu_str.replace(',', '.')
    bagian = waktu_str.split(':')
    return (int(bagian[0]) * 3600) + (int(bagian[1]) * 60) + float(bagian[2])

def format_detik_ke_teks(detik):
    return str(timedelta(seconds=int(detik)))

def bersihkan_teks(teks):
    teks = re.sub(r'<[^>]+>', '', teks)
    teks = " ".join(teks.split())
    return teks

# 1. LOGIN KE GOOGLE
print("🔐 Menghubungkan ke Akun Google Anda...")
auth.authenticate_user()
creds, _ = default()
client = gspread.authorize(creds)

# 2. DOWNLOAD SUBTITLE
ydl_opts = {
    'skip_download': True,
    'writesubtitles': True,
    'writeautomaticsub': True,
    'subtitleslangs': ['id'],
    'outtmpl': f'{nama_file_sementara}.%(ext)s',
    'quiet': True,
    'no_warnings': True
}

print("⏳ Mengambil subtitle dari YouTube...")
try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([link_youtube])
except Exception as e:
    print(f"❌ Gagal download: {e}")

# 3. PROSES DATA
file_terdownload = None
for file in os.listdir('.'):
    if file.startswith(nama_file_sementara) and (file.endswith('.vtt') or file.endswith('.srt')):
        file_terdownload = file
        break

if not file_terdownload:
    print("❌ Subtitle tidak ditemukan.")
else:
    print("⚙️ Memproses teks...")
    with open(file_terdownload, 'r', encoding='utf-8') as f:
        baris_teks = f.readlines()

    chunks = {}
    teks_akumulasi_blok = ""

    for i in range(len(baris_teks)):
        line = baris_teks[i].strip()
        if "-->" in line:
            waktu_mulai_str = line.split("-->")[0].strip()
            try:
                detik_mulai = waktu_ke_detik(waktu_mulai_str)
                indeks_interval = int(detik_mulai // 30)
                if indeks_interval not in chunks: chunks[indeks_interval] = []

                j = i + 1
                current_block_text = ""
                while j < len(baris_teks) and baris_teks[j].strip() != "" and "-->" not in baris_teks[j]:
                    current_block_text += " " + baris_teks[j].strip()
                    j += 1

                current_block_text = bersihkan_teks(current_block_text)

                if current_block_text:
                    if teks_akumulasi_blok and current_block_text.startswith(teks_akumulasi_blok):
                        new_part = current_block_text[len(teks_akumulasi_blok):].strip()
                        if new_part: chunks[indeks_interval].append(new_part)
                    elif current_block_text not in teks_akumulasi_blok:
                        chunks[indeks_interval].append(current_block_text)
                    teks_akumulasi_blok = current_block_text
            except:
                continue

    # 4. PENYUSUNAN DATA
    data_final = [['Interval', 'Mulai', 'Selesai', 'Teks Subtitle']]
    for indeks in sorted(chunks.keys()):
        gabungan = " ".join(chunks[indeks])
        gabungan = re.sub(r'\b(\w+)( \1\b)+', r'\1', gabungan)
        if gabungan.strip():
            data_final.append([indeks + 1, format_detik_ke_teks(indeks * 30), format_detik_ke_teks((indeks + 1) * 30), gabungan.strip()])

    # 5. UPLOAD & FORMATTING (METODE PELURU PERAK)
    try:
        try:
            sh = client.open(nama_sheet_tujuan)
        except gspread.SpreadsheetNotFound:
            sh = client.create(nama_sheet_tujuan)

        worksheet = sh.get_worksheet(0)
        worksheet.clear()

        # Update data
        worksheet.update(values=data_final, range_name='A1')

        # --- JALUR LANGSUNG UNTUK FORMATTING ---
        # Kita pakai batch_update untuk setting lebar kolom dan wrap text sekaligus
        requests = [
            # 1. Setting Lebar Kolom D (index 3)
            {
                "updateDimensionProperties": {
                    "range": {
                        "sheetId": worksheet.id,
                        "dimension": "COLUMNS",
                        "startIndex": 3, # Kolom D
                        "endIndex": 4
                    },
                    "properties": {"pixelSize": 600},
                    "fields": "pixelSize"
                }
            },
            # 2. Setting Wrap Text & Alignment untuk Kolom D
            {
                "repeatCell": {
                    "range": {
                        "sheetId": worksheet.id,
                        "startColumnIndex": 3,
                        "endColumnIndex": 4
                    },
                    "cell": {
                        "userEnteredFormat": {
                            "wrapStrategy": "WRAP",
                            "verticalAlignment": "TOP"
                        }
                    },
                    "fields": "userEnteredFormat(wrapStrategy, verticalAlignment)"
                }
            }
        ]
        sh.batch_update({"requests": requests})

        print(f"\n✅ BERHASIL TOTAL!")
        print(f"🔗 Link: https://docs.google.com/spreadsheets/d/{sh.id}")

    except Exception as e:
        print(f"❌ Terjadi kendala saat upload: {e}")

    if os.path.exists(file_terdownload): os.remove(file_terdownload)

🔐 Menghubungkan ke Akun Google Anda...
⏳ Mengambil subtitle dari YouTube...


ERROR: [youtube] 5iW_mfLSSW4: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


❌ Gagal download: ERROR: [youtube] 5iW_mfLSSW4: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
❌ Subtitle tidak ditemukan.


In [ ]:
import yt_dlp

# 1. Masukkan Link YouTube di sini
link_youtube = "https://www.youtube.com/live/QvZ3MHVs9OM?si=B7K6gWisGuAOnjQv"

# 2. Pengaturan Download Subtitle
ydl_opts = {
    'skip_download': True,        # Abaikan videonya, kita HANYA butuh subtitle
    'writesubtitles': True,       # Ambil subtitle resmi (jika ada)
    'writeautomaticsub': True,    # Ambil subtitle otomatis/auto-generated (jika tidak ada yg resmi)
    'subtitleslangs': ['id'],     # Target bahasa: Indonesia ('id')
    'subtitlesformat': 'srt',     # Format yang diinginkan: SRT
    'outtmpl': '%(title)s.%(ext)s'# Nama file SRT yang didownload akan mengikuti Judul Video
}

# 3. Proses Eksekusi
print(f"Mencoba mendownload subtitle dari: {link_youtube}")
try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([link_youtube])
    print("\n✅ SELESAI! File subtitle (.id.srt atau .id.vtt) berhasil didownload dan tersimpan di folder Anda.")
except Exception as e:
    print(f"\n❌ Terjadi kesalahan: {e}")

In [ ]:
from google.colab import auth
import gspread
from google.auth import default

# 1. Login ke akun Google Anda (akan muncul pop-up izin)
auth.authenticate_user()

# 2. Ambil kredensial otomatis
creds, _ = default()
client = gspread.authorize(creds)

print("✅ Berhasil login menggunakan akun Google Anda!")

✅ Berhasil login menggunakan akun Google Anda!


In [ ]:
import yt_dlp
import pandas as pd
from openpyxl.styles import Font, Alignment, PatternFill
from datetime import timedelta
import os
import re

# ================= PENGATURAN =================
link_youtube = "https://www.youtube.com/live/kL_Xb19YYPQ?si=o-94g74zaLog1Qc5"
nama_file_excel = "Subtitle_30Detik_Final_Akurat.xlsx"
nama_file_sementara = "temp_subtitle"
# ==============================================

def waktu_ke_detik(waktu_str):
    waktu_str = waktu_str.replace(',', '.')
    bagian = waktu_str.split(':')
    return (int(bagian[0]) * 3600) + (int(bagian[1]) * 60) + float(bagian[2])

def format_detik_ke_teks(detik):
    return str(timedelta(seconds=int(detik)))

def bersihkan_teks(teks):
    # Hapus tag HTML seperti <c> atau </c>
    teks = re.sub(r'<[^>]+>', '', teks)
    # Hapus spasi berlebih
    teks = " ".join(teks.split())
    return teks

# 1. DOWNLOAD SUBTITLE
ydl_opts = {
    'skip_download': True,
    'writesubtitles': True,
    'writeautomaticsub': True,
    'subtitleslangs': ['id'],
    'outtmpl': f'{nama_file_sementara}.%(ext)s',
    'quiet': True,
    'no_warnings': True
}

print("⏳ Sedang mengambil subtitle...")
try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([link_youtube])
except Exception as e:
    print(f"❌ Gagal download: {e}")

# 2. MENCARI FILE
file_terdownload = None
for file in os.listdir('.'):
    if file.startswith(nama_file_sementara) and (file.endswith('.vtt') or file.endswith('.srt')):
        file_terdownload = file
        break

if not file_terdownload:
    print("❌ Subtitle tidak ditemukan.")
else:
    # 3. PROSES DENGAN LOGIKA PENYAMBUNGAN KALIMAT (OVERLAP DETECTION)
    with open(file_terdownload, 'r', encoding='utf-8') as f:
        baris_teks = f.readlines()

    chunks = {}
    teks_akumulasi_blok = ""

    for i in range(len(baris_teks)):
        line = baris_teks[i].strip()

        if "-->" in line:
            waktu_mulai_str = line.split("-->")[0].strip()
            try:
                detik_mulai = waktu_ke_detik(waktu_mulai_str)
                indeks_interval = int(detik_mulai // 30)

                if indeks_interval not in chunks:
                    chunks[indeks_interval] = []

                # Ambil teks di bawah timestamp
                j = i + 1
                current_block_text = ""
                while j < len(baris_teks) and baris_teks[j].strip() != "" and "-->" not in baris_teks[j]:
                    current_block_text += " " + baris_teks[j].strip()
                    j += 1

                current_block_text = bersihkan_teks(current_block_text)

                if current_block_text:
                    # LOGIKA UTAMA: Bandingkan dengan teks sebelumnya
                    # Jika teks sekarang sudah diawali oleh teks sebelumnya, ambil sisanya
                    if teks_akumulasi_blok and current_block_text.startswith(teks_akumulasi_blok):
                        new_part = current_block_text[len(teks_akumulasi_blok):].strip()
                        if new_part:
                            chunks[indeks_interval].append(new_part)
                    elif current_block_text not in teks_akumulasi_blok:
                        # Jika benar-benar baru atau berbeda, masukkan semua
                        chunks[indeks_interval].append(current_block_text)

                    teks_akumulasi_blok = current_block_text
            except:
                continue

    # 4. PENYUSUNAN DATA EXCEL
    data_final = []
    for indeks in sorted(chunks.keys()):
        gabungan = " ".join(chunks[indeks])
        # Hapus duplikasi kata yang berdampingan (gagap)
        gabungan = re.sub(r'\b(\w+)( \1\b)+', r'\1', gabungan)

        if gabungan.strip():
            data_final.append([
                indeks + 1,
                format_detik_ke_teks(indeks * 30),
                format_detik_ke_teks((indeks + 1) * 30),
                gabungan.strip()
            ])

    # 5. SIMPAN KE EXCEL
    df = pd.DataFrame(data_final, columns=['Interval', 'Mulai', 'Selesai', 'Teks Subtitle'])
    with pd.ExcelWriter(nama_file_excel, engine='openpyxl') as writer:
        df.to_excel(writer, index=False)
        ws = writer.sheets['Sheet1']
        ws.column_dimensions['D'].width = 120
        for cell in ws['D']:
            cell.alignment = Alignment(wrap_text=True, vertical='top')

    print(f"🎉 SELESAI! Hasil lebih akurat di: {nama_file_excel}")
    if os.path.exists(file_terdownload): os.remove(file_terdownload)

⏳ Sedang mengambil subtitle...
🎉 SELESAI! Hasil lebih akurat di: Subtitle_30Detik_Final_Akurat.xlsx
